In [1]:
import os
print(os.getcwd())
print(os.path.exists("app/cnn_model.py"))

/Users/kellylyons/sps_genai
True


In [3]:
import os

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from tqdm import tqdm

from app.cnn_model import SimpleCNN

torch.manual_seed(42)

EPOCHS = 10
BATCH_SIZE = 32

device = (
    torch.device("mps")
    if torch.backends.mps.is_available()
    else torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
)
print(f"Using device: {device}")

# Resize 32x32 CIFAR images to 64x64 (the input size our CNN expects)
transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
])

train_dataset = torchvision.datasets.CIFAR10(root="./data", train=True, download=True, transform=transform)
test_dataset = torchvision.datasets.CIFAR10(root="./data", train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

model = SimpleCNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0005)

# ---- Training ----
for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    running_correct, running_total = 0, 0

    for inputs, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}"):
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        _, predicted = torch.max(outputs.data, 1)
        running_correct += (predicted == labels).sum().item()
        running_total += labels.size(0)
        running_loss += loss.item()

    print(f"Epoch {epoch+1}: loss {running_loss/len(train_loader):.4f}, "
          f"train accuracy {running_correct/running_total:.3f}")

# ---- Evaluation ----
model.eval()
test_correct, test_total = 0, 0
with torch.no_grad():
    for test_images, test_labels in test_loader:
        test_images, test_labels = test_images.to(device), test_labels.to(device)
        _, test_predicted = torch.max(model(test_images).data, 1)
        test_total += test_labels.size(0)
        test_correct += (test_predicted == test_labels).sum().item()
print(f"Test accuracy: {100 * test_correct / test_total:.2f}%")

# ---- Save the trained weights for the API ----
os.makedirs("app/models", exist_ok=True)
torch.save(model.cpu().state_dict(), "app/models/cnn_cifar10.pth")
print("Saved model to app/models/cnn_cifar10.pth")

Using device: mps


100%|██████████| 170M/170M [51:09<00:00, 55.5kB/s]   
Epoch 1/10: 100%|██████████| 1563/1563 [00:27<00:00, 56.24it/s]


Epoch 1: loss 1.5801, train accuracy 0.435


Epoch 2/10: 100%|██████████| 1563/1563 [00:25<00:00, 60.45it/s]


Epoch 2: loss 1.2695, train accuracy 0.546


...

Epoch 8: loss 0.7661, train accuracy 0.730


Epoch 9/10: 100%|██████████| 1563/1563 [00:25<00:00, 60.86it/s]


Epoch 9: loss 0.7116, train accuracy 0.751


Epoch 10/10: 100%|██████████| 1563/1563 [00:25<00:00, 60.39it/s]


Epoch 10: loss 0.6607, train accuracy 0.768
Test accuracy: 66.23%
Saved model to app/models/cnn_cifar10.pth
